<div class="nb-header">
  <span class="nb-header__type">Tutorial</span>
  <h1 class="nb-header__title">Handling Errors</h1>
  <p class="nb-header__subtitle">Understand the exception hierarchy and implement robust error handling</p>
  <div class="nb-header__meta">
    <span class="nb-header__meta-item nb-header__meta-item--duration">20 min</span>
    <span class="nb-header__meta-item nb-header__meta-item--level">
      <span class="nb-difficulty nb-difficulty--intermediate">
        <span class="nb-difficulty__dot"></span>
        <span class="nb-difficulty__dot"></span>
        <span class="nb-difficulty__dot"></span>
      </span>
      Intermediate
    </span>
  </div>
  <div class="nb-header__tags"><span class="nb-header__tag">Errors</span><span class="nb-header__tag">Exceptions</span><span class="nb-header__tag">Debugging</span><span class="nb-header__tag">Retry</span></div>
</div>

<div class="nb-objectives">
  <h3 class="nb-objectives__title">What You'll Learn</h3>
  <ul class="nb-objectives__list">
    <li><strong>Exception Hierarchy</strong> - Understand GraphOLAPError subtypes</li>
    <li><strong>Common Errors</strong> - Handle authentication, not found, validation errors</li>
    <li><strong>Retry Logic</strong> - Implement retry for transient failures</li>
    <li><strong>Debugging</strong> - Use error details for troubleshooting</li>
  </ul>
</div>

<div class="nb-section">
  <span class="nb-section__number">1</span>
  <div>
    <h2 class="nb-section__title">Exception Hierarchy</h2>
    <p class="nb-section__description">GraphOLAPError and subtypes</p>
  </div>
</div>

In [ ]:
from graph_olap.exceptions import (
    GraphOLAPError,
    AuthenticationError,
    ResourceNotFoundError,
    ValidationError,
    RateLimitError,
    ServerError
)

# Hierarchy:
# GraphOLAPError
#   ├── AuthenticationError
#   ├── ResourceNotFoundError
#   ├── ValidationError
#   ├── RateLimitError
#   └── ServerError

<div class="nb-section">
  <span class="nb-section__number">2</span>
  <div>
    <h2 class="nb-section__title">Authentication Errors</h2>
    <p class="nb-section__description">Handle auth failures</p>
  </div>
</div>

In [ ]:
try:
    mapping = client.mappings.get(mapping_id)
except AuthenticationError:
    print("Check your API key")
    # Re-authenticate or prompt user
except ResourceNotFoundError:
    print(f"Mapping {mapping_id} not found")
    # Create new or use different ID
except ValidationError as e:
    print(f"Invalid request: {e.details}")
except GraphOLAPError as e:
    print(f"SDK error: {e}")

<div class="nb-section">
  <span class="nb-section__number">3</span>
  <div>
    <h2 class="nb-section__title">Resource Errors</h2>
    <p class="nb-section__description">Not found and validation errors</p>
  </div>
</div>

In [ ]:
import time
from graph_olap.exceptions import RateLimitError, ServerError

def with_retry(func, max_retries=3):
    for attempt in range(max_retries):
        try:
            return func()
        except (RateLimitError, ServerError) as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt  # Exponential backoff
            print(f"Retry in {wait}s...")
            time.sleep(wait)

# Usage
result = with_retry(lambda: client.mappings.list())

<div class="nb-section">
  <span class="nb-section__number">4</span>
  <div>
    <h2 class="nb-section__title">Implementing Retry</h2>
    <p class="nb-section__description">Handle transient failures</p>
  </div>
</div>

In [ ]:
try:
    mapping = client.mappings.create(...)
except ValidationError as e:
    print(f"Error: {e.message}")
    print(f"Details: {e.details}")
    print(f"Request ID: {e.request_id}")  # For support

<div class="nb-takeaways">
  <h3 class="nb-takeaways__title">Key Takeaways</h3>
  <ul class="nb-takeaways__list">
    <li>All SDK errors inherit from GraphOLAPError</li>
    <li>AuthenticationError for invalid/expired credentials</li>
    <li>ResourceNotFoundError when resource doesn't exist</li>
    <li>Use exponential backoff for transient failures</li>
  </ul>
</div>